# MAINTAIN AI V1.4 — Public Predictive-Maintenance Model Benchmark

This notebook implements the public architecture family used by the IEEE IES Industrial Predictive Maintenance reference project: **LSTM + attention, Transformer, TCN, and LSTM autoencoder**. The reference project documents these four models for RUL/fault/anomaly tasks and provides a unified API. urlIEEE IES Industrial Predictive Maintenancehttps://github.com/IEEE-IES-Industrial-AI-Lab/Industrial-Predictive-Maintenance

**Important:** this notebook does not pretend public C-MAPSS models are compressor/motor models. We train the architectures on MAINTAIN AI's existing MetroPT-3 compressor risk sequences using the existing event-aware train/validation/test split. The goal is a fair architecture benchmark before we build MAINTAIN AI's own model.

Foundation-model candidates such as TimeRadar and Chronos are kept as a separate adapter stage because their native interfaces/tasks differ from our 24-step multivariate future-risk head. TimeRadar provides a ready-to-use pretrained checkpoint for anomaly detection. urlTimeRadarhttps://github.com/mala-lab/TimeRadar

In [ ]:
!pip -q install numpy pandas scikit-learn torch pyarrow
import os, json, math, time, copy, random, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=Path('/content/maintain_ai_v1_3')
SEQ=ROOT/'temporal_sequences_v1_3'
ART=ROOT/'artifacts_public_models_v1_4'; ART.mkdir(parents=True,exist_ok=True)
print('device:',DEVICE)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 1. Load the same leakage-safe MetroPT-3 split

We reuse the V1.3 sequence tensors already created in Colab. No new labels are invented here. The model sees a 24-step history and predicts the existing 24h/48h/7d future-risk targets.

In [ ]:
X=np.load(SEQ/'metro_X.npy',mmap_mode='r')
Y=np.load(SEQ/'metro_y.npy',mmap_mode='r').astype(np.float32)
T=np.load(SEQ/'metro_times.npy',allow_pickle=True)
splits=np.load(SEQ/'metro_splits.npz') if (SEQ/'metro_splits.npz').exists() else None
if splits is None:
    # Reconstruct from the event-aware timestamps used by V1.3.
    ts=pd.to_datetime(T,utc=True)
    test_start=pd.Timestamp('2020-07-08 14:30:00',tz='UTC'); test_end=pd.Timestamp('2020-07-15 14:30:00',tz='UTC')
    val_start=pd.Timestamp('2020-05-29 10:00:00',tz='UTC'); val_end=pd.Timestamp('2020-06-05 10:00:00',tz='UTC')
    test=np.flatnonzero((ts>=test_start)&(ts<test_end)); val=np.flatnonzero((ts>=val_start)&(ts<val_end)); used=np.union1d(val,test); train=np.setdiff1d(np.arange(len(ts)),used)
    np.savez(SEQ/'metro_splits.npz',train=train,val=val,test=test); splits=np.load(SEQ/'metro_splits.npz')
tr,va,te=[splits[k] for k in ('train','val','test')]
print('X',X.shape,'Y',Y.shape,'train/val/test',len(tr),len(va),len(te))
print('positive rates:',Y[tr].mean(0),Y[va].mean(0),Y[te].mean(0))

In [ ]:
# Fit normalization on training assets/windows only. Chunked to avoid a giant RAM copy.
sum_x=np.zeros(X.shape[-1],dtype=np.float64); sum_x2=np.zeros(X.shape[-1],dtype=np.float64); n=0
for start in range(0,len(tr),2048):
    a=np.asarray(X[tr[start:start+2048]],dtype=np.float32)
    flat=a.reshape(-1,a.shape[-1]); sum_x+=flat.sum(0); sum_x2+=(flat.astype(np.float64)**2).sum(0); n+=len(flat)
mu=(sum_x/n).astype(np.float32); sd=np.sqrt(np.maximum(sum_x2/n-(sum_x/n)**2,1e-8)).astype(np.float32)
np.savez(ART/'metro_normalization.npz',mean=mu,std=sd)
print('normalization saved; zero/near-zero std:',int((sd<1e-4).sum()))

In [ ]:
class MetroDS(Dataset):
    def __init__(self,idx): self.idx=np.asarray(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=int(self.idx[i]); x=np.array(X[j],dtype=np.float32); x=(x-mu)/(sd+1e-6); y=np.array(Y[j],dtype=np.float32)
        return torch.from_numpy(x),torch.from_numpy(y)

train_loader=DataLoader(MetroDS(tr),batch_size=256,shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available(),persistent_workers=True)
val_loader=DataLoader(MetroDS(va),batch_size=512,shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available(),persistent_workers=True)
test_loader=DataLoader(MetroDS(te),batch_size=512,shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available(),persistent_workers=True)
IN_DIM=X.shape[-1]; SEQ_LEN=X.shape[1]; H=128
print(IN_DIM,SEQ_LEN)

## 2. Public architecture implementations

These follow the architecture classes described by the public reference: stacked LSTM + Bahdanau-style attention, encoder Transformer with positional encoding, dilated causal TCN, and LSTM autoencoder. We adapt the predictive models to MAINTAIN AI's three future-risk logits instead of a scalar C-MAPSS RUL output.

In [ ]:
class AttentionLSTM(nn.Module):
    def __init__(self,d=128):
        super().__init__(); self.inp=nn.Linear(IN_DIM,d); self.lstm=nn.LSTM(d,d,num_layers=2,batch_first=True,dropout=.1)
        self.q=nn.Linear(d,d); self.k=nn.Linear(d,d); self.v=nn.Linear(d,d); self.out=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,3))
    def forward(self,x):
        h,_=self.lstm(self.inp(x)); q=h[:,-1:]; e=torch.tanh(self.q(q)+self.k(h)); a=torch.softmax(self.v(e).squeeze(-1),dim=1); c=(a.unsqueeze(-1)*h).sum(1); return self.out(c)

class TransformerRisk(nn.Module):
    def __init__(self,d=128,heads=8,layers=3):
        super().__init__(); self.inp=nn.Linear(IN_DIM,d); self.pos=nn.Parameter(torch.zeros(1,SEQ_LEN,d)); nn.init.normal_(self.pos,std=.02)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=4*d,dropout=.1,batch_first=True,norm_first=True)
        self.enc=nn.TransformerEncoder(layer,num_layers=layers); self.out=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,3))
    def forward(self,x): return self.out(self.enc(self.inp(x)+self.pos)[:,-1])

class Chomp1d(nn.Module):
    def __init__(self,n): super().__init__(); self.n=n
    def forward(self,x): return x[:,:,:-self.n] if self.n else x

class TCNBlock(nn.Module):
    def __init__(self,cin,cout,dil):
        super().__init__(); k=3; p=(k-1)*dil
        self.net=nn.Sequential(nn.Conv1d(cin,cout,k,padding=p,dilation=dil),Chomp1d(p),nn.GELU(),nn.GroupNorm(8,cout),nn.Conv1d(cout,cout,k,padding=p,dilation=dil),Chomp1d(p),nn.GELU(),nn.GroupNorm(8,cout))
        self.skip=nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()
    def forward(self,x): return self.net(x)+self.skip(x)

class TCNRisk(nn.Module):
    def __init__(self,d=128):
        super().__init__(); self.proj=nn.Conv1d(IN_DIM,d,1); self.net=nn.Sequential(TCNBlock(d,d,1),TCNBlock(d,d,2),TCNBlock(d,d,4),TCNBlock(d,d,8),TCNBlock(d,d,16)); self.out=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,3))
    def forward(self,x): return self.out(self.net(self.proj(x.transpose(1,2)))[:,:,-1])

MODELS={'lstm_attention':AttentionLSTM(),'transformer':TransformerRisk(),'tcn':TCNRisk()}
for n,m in MODELS.items(): print(n,sum(p.numel() for p in m.parameters()))

In [ ]:
pos=Y[tr].sum(0); neg=len(tr)-pos; POS_W=torch.tensor((neg/np.maximum(pos,1)),dtype=torch.float32,device=DEVICE)
print('positive weights:',POS_W.detach().cpu().numpy())

def evaluate(model,loader):
    model.eval(); ys=[]; ps=[]; losses=[]
    crit=nn.BCEWithLogitsLoss(pos_weight=POS_W,reduction='none')
    with torch.no_grad():
        for xb,yb in loader:
            z=model(xb.to(DEVICE,non_blocking=True)); y=yb.to(DEVICE,non_blocking=True); losses.append(crit(z,y).mean().item()); ys.append(y.cpu().numpy()); ps.append(torch.sigmoid(z).cpu().numpy())
    y=np.concatenate(ys); p=np.concatenate(ps); auc=[]; ap=[]; f=[]
    for k in range(3):
        if y[:,k].min()==y[:,k].max(): auc.append(np.nan); ap.append(np.nan); f.append(np.nan); continue
        auc.append(roc_auc_score(y[:,k],p[:,k])); ap.append(average_precision_score(y[:,k],p[:,k])); pr,rc,f1,_=precision_recall_fscore_support(y[:,k],p[:,k]>=.5,average='binary',zero_division=0); f.append(f1)
    return {'loss':float(np.mean(losses)),'roc_auc':auc,'pr_auc':ap,'f1_at_0_5':f,'positive_rate':y.mean(0).tolist()}

def train_model(name,model,epochs=15,patience=4):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4); crit=nn.BCEWithLogitsLoss(pos_weight=POS_W)
    best=-np.inf; best_state=None; wait=0; hist=[]
    for ep in range(1,epochs+1):
        model.train(); running=[]; t0=time.time()
        for xb,yb in train_loader:
            xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); loss=crit(model(xb),yb); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step(); running.append(loss.item())
        vm=evaluate(model,val_loader); score=np.nanmean(vm['pr_auc'][:2])
        hist.append({'epoch':ep,'train_loss':float(np.mean(running)),'val':vm,'seconds':time.time()-t0}); print(name,ep,'train',round(np.mean(running),4),'val_loss',round(vm['loss'],4),'PR-AUC24/48',np.round(vm['pr_auc'][:2],4))
        if score>best: best=score; best_state=copy.deepcopy(model.state_dict()); wait=0
        else: wait+=1
        if wait>=patience: break
    model.load_state_dict(best_state)
    test=evaluate(model,test_loader)
    torch.save({'model':model.state_dict(),'model_name':name,'input_dim':IN_DIM,'sequence_length':SEQ_LEN,'normalization':'metro_normalization.npz'},ART/f'{name}.pt')
    json.dump({'best_val_mean_pr_auc_24_48':float(best),'history':hist,'test':test},open(ART/f'{name}_metrics.json','w'),indent=2)
    return test

results={}
for name,model in MODELS.items(): results[name]=train_model(name,model)
print(json.dumps(results,indent=2))

## 3. Unsupervised anomaly model

The public reference uses an LSTM autoencoder for anomaly detection. We train it only on historical normal compressor windows, then score the held-out pre-event windows. This is deliberately a **separate anomaly signal**, not relabeled as future failure probability.

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self,d=96):
        super().__init__(); self.enc=nn.LSTM(IN_DIM,d,num_layers=2,batch_first=True); self.dec=nn.LSTM(d,d,num_layers=2,batch_first=True); self.proj=nn.Linear(d,IN_DIM)
    def forward(self,x):
        _,(h,c)=self.enc(x); z=h[-1].unsqueeze(1).repeat(1,x.size(1),1); y,_=self.dec(z,(h,c)); return self.proj(y)

# Keep training normal: no future-risk positive in any target horizon and not inside a known event lead-up.
normal_tr=tr[Y[tr].max(1)==0]
ae_loader=DataLoader(MetroDS(normal_tr),batch_size=256,shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available(),persistent_workers=True)
ae=LSTMAutoencoder().to(DEVICE); opt=torch.optim.AdamW(ae.parameters(),lr=3e-4,weight_decay=1e-4); mse=nn.MSELoss()
best=np.inf; best_state=None
for ep in range(1,11):
    ae.train(); ls=[]
    for xb,_ in ae_loader:
        xb=xb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); loss=mse(ae(xb),xb); loss.backward(); nn.utils.clip_grad_norm_(ae.parameters(),2); opt.step(); ls.append(loss.item())
    ae.eval(); vs=[]
    with torch.no_grad():
        for xb,_ in val_loader:
            xb=xb.to(DEVICE,non_blocking=True); vs.append(mse(ae(xb),xb).item())
    v=float(np.mean(vs)); print('AE',ep,'train',round(float(np.mean(ls)),5),'val',round(v,5))
    if v<best: best=v; best_state=copy.deepcopy(ae.state_dict())
ae.load_state_dict(best_state); torch.save({'model':ae.state_dict(),'input_dim':IN_DIM,'sequence_length':SEQ_LEN},ART/'lstm_autoencoder.pt')
print('AE saved')

In [ ]:
def anomaly_scores(model,loader):
    model.eval(); out=[]
    with torch.no_grad():
        for xb,_ in loader:
            xb=xb.to(DEVICE,non_blocking=True); r=(model(xb)-xb).pow(2).mean((1,2)); out.append(r.cpu().numpy())
    return np.concatenate(out)
val_score=anomaly_scores(ae,val_loader); test_score=anomaly_scores(ae,test_loader)
threshold=float(np.quantile(val_score,0.99)); test_labels=Y[te,0]
print('threshold (99th percentile validation):',threshold)
if len(np.unique(test_labels))>1:
    print('test anomaly ROC-AUC vs 24h future-risk:',roc_auc_score(test_labels,test_score),'PR-AUC:',average_precision_score(test_labels,test_score))
print('note: this is an anomaly-vs-future-risk diagnostic, not a claim that reconstruction error equals failure probability.')

## 4. Save a single benchmark manifest

The next MAINTAIN AI model should beat or justify these public baselines on the same split. We will not select a model from training loss alone.

In [ ]:
manifest={
  'version':'maintain-ai-public-model-benchmark-v1.4',
  'reference':'IEEE-IES-Industrial-AI-Lab/Industrial-Predictive-Maintenance',
  'architectures':['lstm_attention','transformer','tcn','lstm_autoencoder'],
  'dataset':'MetroPT-3 compressor',
  'input_shape':[int(SEQ_LEN),int(IN_DIM)],
  'targets':['future_risk_24h','future_risk_48h','future_risk_7d'],
  'split':'existing event-aware V1.3 train/validation/test',
  'normalization':'training split only',
  'artifacts':sorted(p.name for p in ART.iterdir())
}
json.dump(manifest,open(ART/'manifest.json','w'),indent=2)
print(json.dumps(manifest,indent=2))